# Refactoring a Jupyter Notebook into a Python Script

This notebook shows how to move from **exploratory notebook code** to **reusable Python functions** using the King County Housing dataset.

This is a common real-world handoff:
- data scientists explore the data and test ideas in notebook cells,
- ML engineers identify the stable transformations,
- and those transformations are rewritten as importable functions.

The goal is not to remove exploration. The goal is to separate:
1. **one-off analysis** that helps us understand the data, and
2. **repeatable preprocessing logic** that should live in Python modules.

This notebook focuses on the reasoning behind the refactor. The companion notebook `king-county-data-preparation.ipynb` shows the function-extraction step by step.


> **Checkpoint:**
> You can identify which cells are exploratory and which should become reusable functions.

> **Common pitfalls:**
> - Copying notebook code into a script without simplifying it.
> - Keeping one-off analysis statements inside reusable preprocessing functions.
> - Forgetting to define a clear DataFrame-in/DataFrame-out contract.

> **Self-check:**
> Could another teammate import your refactored functions without needing notebook state or hidden variables?


## Visual Guide: Refactor Strategy

```mermaid
flowchart TD
    A["Exploratory notebook cells"]
    B["Identify stable cleaning rules"]
    C["Extract focused functions"]
    D["Move logic into a Python module"]
    E["Reuse outside the notebook"]

    A --> B --> C --> D --> E
```


In [1]:
# pandas is used for tabular data exploration and cleanup.
import pandas as pd

In [2]:
# Load the raw housing dataset from the shared repository data folder.
# We start in the notebook because this is still the exploratory phase.
df = pd.read_csv("../data/King_County_House_prices_dataset.csv")

# Display a sample so we can sanity-check the columns and raw values.
df.head(10)

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,10/13/2014,221900.0,3,1.00,1180,5650,1.0,NaN,0.0,...,7,1180,0.0,1955,0.0,98178,47.5112,-122.257,1340,5650
1,6414100192,12/9/2014,538000.0,3,2.25,2570,7242,2.0,0.0,0.0,...,7,2170,400.0,1951,1991.0,98125,47.7210,-122.319,1690,7639
2,5631500400,2/25/2015,180000.0,2,1.00,770,10000,1.0,0.0,0.0,...,6,770,0.0,1933,NaN,98028,47.7379,-122.233,2720,8062
3,2487200875,12/9/2014,604000.0,4,3.00,1960,5000,1.0,0.0,0.0,...,7,1050,910.0,1965,0.0,98136,47.5208,-122.393,1360,5000
4,1954400510,2/18/2015,510000.0,3,2.00,1680,8080,1.0,0.0,0.0,...,8,1680,0.0,1987,0.0,98074,47.6168,-122.045,1800,7503
5,7237550310,5/12/2014,1230000.0,4,4.50,5420,101930,1.0,0.0,0.0,...,11,3890,1530.0,2001,0.0,98053,47.6561,-122.005,4760,101930
6,1321400060,6/27/2014,257500.0,3,2.25,1715,6819,2.0,0.0,0.0,...,7,1715,?,1995,0.0,98003,47.3097,-122.327,2238,6819
7,2008000270,1/15/2015,291850.0,3,1.50,1060,9711,1.0,0.0,NaN,...,7,1060,0.0,1963,0.0,98198,47.4095,-122.315,1650,9711
8,2414600126,4/15/2015,229500.0,3,1.00,1780,7470,1.0,0.0,0.0,...,7,1050,730.0,1960,0.0,98146,47.5123,-122.337,1780,8113
9,3793500160,3/12/2015,323000.0,3,2.50,1890,6560,2.0,0.0,0.0,...,7,1890,0.0,2003,0.0,98038,47.3684,-122.031,2390,7570


## Data Preparation

Before we write reusable Python functions, we need to understand the dataset well enough to decide **which transformations are stable and worth keeping**.

The questions below are still part of the exploratory phase. They help us decide:
- what the data quality issues are,
- which rules are defensible,
- and which pieces of logic should later move into a Python module.


### Visual Guide: What Moves Out of the Notebook?

```mermaid
flowchart TD
    A["Notebook cell"]
    B{"Is the logic repeatable?"}
    C["Keep it in notebook"]
    D["Turn it into a function"]
    E["Move it into a module"]

    A --> B
    B -->|No| C
    B -->|Yes| D --> E
```


**1.**  How many house sales are in our dataset?


In [3]:
# `shape` returns `(rows, columns)`.
# This is a quick way to understand the overall size of the dataset.
df.shape

(21597, 21)

Each row contains the details of one house sale --> **21597** house sales are included in our dataset.


**2.** What format is the data in (**int** = number, **float** = decimal, **str** = text)?
Which formats are surprising?


In [4]:
# `.info()` gives us a schema summary: column names, non-null counts, and data types.
# This is one of the fastest ways to spot unexpected string columns or missing values.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21597 non-null  int64  
 1   date           21597 non-null  str    
 2   price          21597 non-null  float64
 3   bedrooms       21597 non-null  int64  
 4   bathrooms      21597 non-null  float64
 5   sqft_living    21597 non-null  int64  
 6   sqft_lot       21597 non-null  int64  
 7   floors         21597 non-null  float64
 8   waterfront     19221 non-null  float64
 9   view           21534 non-null  float64
 10  condition      21597 non-null  int64  
 11  grade          21597 non-null  int64  
 12  sqft_above     21597 non-null  int64  
 13  sqft_basement  21597 non-null  str    
 14  yr_built       21597 non-null  int64  
 15  yr_renovated   17755 non-null  float64
 16  zipcode        21597 non-null  int64  
 17  lat            21597 non-null  float64
 18  long           21

Our dataset contains 8 columns with decimal numbers (float), 11 columns with integers (int), and 2 columns with text (str/object).  

We expect all data on the size, number of rooms, the indices, and the information on the location to be recognized as numbers.  

The `date` is recognized as text, as well as the area of the basement (`sqft_basement`). This is surprising! A little further up, when we looked at the first 10 rows of our table, we could see why the values are not recognized as numbers: there is a `?` mixed in.  


**3.** Are there any missing values?


In [5]:
# Count the missing values in each column.
# We use this to decide which columns need a cleanup strategy later.
df.isna().sum()

id                  0
date                0
price               0
bedrooms            0
bathrooms           0
sqft_living         0
sqft_lot            0
floors              0
waterfront       2376
view               63
condition           0
grade               0
sqft_above          0
sqft_basement       0
yr_built            0
yr_renovated     3842
zipcode             0
lat                 0
long                0
sqft_living15       0
sqft_lot15          0
dtype: int64

We have missing values in the columns `waterfront`, `view`, and `yr_renovated`, with the most missing values being in the `yr_renovated` column.  

Further on, we have to **think about how to deal with these missing values**.  


**4.** How many values are there for the individual variables?


In [6]:
# `nunique()` shows how many distinct values each column contains.
# This helps us notice repeated IDs, low-cardinality flags, and suspicious variables.
df.nunique()

id               21420
date               372
price             3622
bedrooms            12
bathrooms           29
sqft_living       1034
sqft_lot          9776
floors               6
waterfront           2
view                 5
condition            5
grade               11
sqft_above         942
sqft_basement      304
yr_built           116
yr_renovated        70
zipcode             70
lat               5033
long               751
sqft_living15      777
sqft_lot15        8682
dtype: int64

**Interesting findings:**  

**1.** There are 21420 different house IDs of 21597 rows in the dataframe : this means that up to 177 houses may have been sold twice.  

**2.** There are 3622 expressions for the price of sold houses: many houses were sold for the same price.  

**3.** Although `grade` is an index from 1-13, there are only 11 different values included.  

**4.** The houses in our dataset were built in 116 different years.  


Next, we can display the **statistical distribution** of the individual columns. Here, too, you are sure to discover a few interesting insights.  


| Stat | Description |
|---|---|
| `count` | Indication of how many values are present in the column (NaN/missing values are not counted) |
| `mean` | Mean value of the data |
| `std` | Standard deviation of the data |
| `min` | The smallest value in the column |
| `25%` | 25% of the data is below this value |
| `50%` | 50% of the data is below this value (this value is called the **median**) |
| `75%` | 75% of the data is below this value |
| `max` | The largest value in the column |


In [9]:
# `describe()` summarizes the numeric columns.
# Rounding makes the output easier to scan while we look for unusual values.
df.describe().round(2)


,id,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
count,2.159700e+04,21597.00,21597.00,21597.00,21597.00,21597.00,21597.00,19221.00,21534.00,21597.00,21597.00,21597.00,21597.00,17755.00,21597.00,21597.00,21597.00,21597.00,21597.00
mean,4.580474e+09,540296.57,3.37,2.12,2080.32,15099.41,1.49,0.01,0.23,3.41,7.66,1788.60,1971.00,83.64,98077.95,47.56,-122.21,1986.62,12758.28
std,2.876736e+09,367368.14,0.93,0.77,918.11,41412.64,0.54,0.09,0.77,0.65,1.17,827.76,29.38,399.95,53.51,0.14,0.14,685.23,27274.44
min,1.000102e+06,78000.00,1.00,0.50,370.00,520.00,1.00,0.00,0.00,1.00,3.00,370.00,1900.00,0.00,98001.00,47.16,-122.52,399.00,651.00
25%,2.123049e+09,322000.00,3.00,1.75,1430.00,5040.00,1.00,0.00,0.00,3.00,7.00,1190.00,1951.00,0.00,98033.00,47.47,-122.33,1490.00,5100.00
50%,3.904930e+09,450000.00,3.00,2.25,1910.00,7618.00,1.50,0.00,0.00,3.00,7.00,1560.00,1975.00,0.00,98065.00,47.57,-122.23,1840.00,7620.00
75%,7.308900e+09,645000.00,4.00,2.50,2550.00,10685.00,2.00,0.00,0.00,4.00,8.00,2210.00,1997.00,0.00,98118.00,47.68,-122.12,2360.00,10083.00
max,9.900000e+09,7700000.00,33.00,8.00,13540.00,1651359.00,3.50,1.00,4.00,5.00,13.00,9410.00,2015.00,2015.00,98199.00,47.78,-121.32,6210.00,871200.00


**Interesting findings:**  

**1.** On average, the houses in this dataset cost \$540,296. The most expensive house sold for \$7,700,000, the cheapest one for \$78,000.  

**2.** 50% of the houses have 3 or fewer bedrooms. The house with the most bedrooms has 33! (Is that possible?)  

**3.** On average, each house has 2.1 bathrooms.  

**4.** 75% of houses have 2 or fewer floors.  

**5.** Only 1% of the houses have a view of the seafront (since `waterfront` can only take the values 0 and 1, the **mean** can be interpreted as the percentage of houses with a view of the seafront).  

**6.** The oldest house in our dataset is from 1900, the newest one from 2015.  


Of course we can find many more insights from the table above, but we have already discovered a few inconsistencies.

These are strong candidates for the later refactor because they represent **repeatable data-cleaning rules**:
- missing values that need an explicit policy,
- `?` in `sqft_basement`, which should be numeric,
- and the suspicious house entry with 33 bedrooms.

The next few cells are intentionally notebook-style and one-off. Later, we will keep only the stable parts and move them into reusable functions.


**33 bedrooms**

If we look at the maximum number of bedrooms, we see a house with 33 bedrooms but less than 2 bathrooms.


In [10]:
# Inspect the suspicious record directly before deciding on a cleaning rule.
# In exploratory work, this kind of one-off query is normal and often useful.
df.query("bedrooms == 33")

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
15856,2402100895,6/25/2014,640000.0,33,1.75,1620,6000,1.0,0.0,0.0,...,7,1040,580.0,1947,0.0,98103,47.6878,-122.331,1330,4700


Since the ratio of 33 bedrooms to 2 bathrooms in 1620 sqft sounds very unlikely, we are removing this house from the record.


In [11]:
# This is still one-off exploratory cleanup code.
# We remove the suspicious row now, then later rewrite the general logic as a reusable function.
df.drop(15856, axis=0, inplace=True)

**`?` in `sqft_basement`**

We have seen in the output of `.info()` that `sqft_basement` is presented as a string and not as a number. This is because there are some `?` in this column, even though basement area should be numeric.  

Because `sqft_living` and `sqft_above` already contain the information we need, it is safer to rebuild the entire `sqft_basement` feature from those two reliable source columns instead of trying to trust the messy raw column.  


In [12]:
# Rebuild basement size directly from two columns that already contain the needed information.
# This is often safer than trusting a raw column that mixes numbers with placeholder strings.
df["sqft_basement"] = df["sqft_living"] - df["sqft_above"]

In [13]:
# Inspect a few recalculated basement values.
df[["sqft_living", "sqft_above", "sqft_basement"]].head()

,sqft_living,sqft_above,sqft_basement
0,1180,1180,0
1,2570,2170,400
2,770,770,0
3,1960,1050,910
4,1680,1680,0


**Missing values**

Before choosing fill rules, we audit how many missing values are left and how large they are relative to the full dataset.


In [17]:
# Build a small audit table with both counts and percentages.
# Percentages help us judge whether missingness is tiny, moderate, or substantial.
missing_values = df.isnull().sum().to_frame(name="count")
#missing_values
missing_values["percentage"] = (missing_values["count"] / df.shape[0] * 100).round(2)
missing_values.query("count != 0")

,count,percentage
waterfront,2376,11.00
view,63,0.29
yr_renovated,3842,17.79


Only 3 features have missing values after rebuilding `sqft_basement` from the reliable source columns.  


In [18]:
# Check the distribution of `view` before deciding how to fill its missing values.
# If one value is overwhelmingly common, a simple fill rule may be reasonable.
df["view"].value_counts()

view
0.0    19421
2.0      957
3.0      508
1.0      330
4.0      317
Name: count, dtype: int64

The column `view` has only 0.29% of missing values and has 19421 out of 21596 times the value 0, so we will replace the missing values in this column with 0.  


In [19]:
# Fill missing `view` values with 0.
# In this workflow, 0 represents "no view" and is also the dominant observed value.
df["view"] = df["view"].fillna(0)

In the column `waterfront` we see a similar distribution:


In [20]:
# Inspect `waterfront` in the same way before choosing a fill value.
df["waterfront"].value_counts()

waterfront
0.0    19074
1.0      146
Name: count, dtype: int64

So also here we replace the NaNs with 0:

In [22]:
# Fill missing `waterfront` values with 0.
# Again, this matches the dominant business meaning in this dataset.
df["waterfront"] = df["waterfront"].fillna(0)

In [23]:
# Rebuild the missing-value summary after the fills above.
# This lets us confirm which columns still need attention.
missing_values = df.isnull().sum().to_frame(name="count")
missing_values["percentage"] = missing_values["count"] / df.shape[0] * 100
missing_values.query("count != 0")

,count,percentage
yr_renovated,3842,17.790332


Since 17.8% of the data is missing from `yr_renovated`, dropping the column outright would discard information and leaving it as-is would keep a messy signal.

A cleaner next step is to create one consolidated feature called `last_known_change`:
- if a house has no renovation year, fall back to `yr_built`,
- otherwise use the renovation year,
- then drop the two original source columns.

This is a good example of logic that belongs in a reusable function because it is deterministic and business-facing.


In [26]:
# Create an empty list to collect the consolidated year values.
last_known_change = []

# Loop through the renovation-year column row by row.
for idx, yr_re in df["yr_renovated"].items():
    # Missing values or 0 mean "no known renovation",
    # so we fall back to the original build year.
    #if str(yr_re) == "nan" or yr_re == 0.0:
    ###better: if pd.isna(yr_re) or yr_re == 0:
    ### yr_re in (0, pd.NA) does not work because pd.NA is not equal to itself.
    if pd.isna(yr_re) or yr_re == 0:
        last_known_change.append(df["yr_built"][idx])
    else:
        # Otherwise, keep the renovation year as the last known change.
        last_known_change.append(int(yr_re))

In [27]:
# Add the consolidated list back as a new feature.
df["last_known_change"] = last_known_change

In [28]:
# Drop the source columns now that their information is captured in `last_known_change`.
df.drop("yr_renovated", axis=1, inplace=True)
df.drop("yr_built", axis=1, inplace=True)

We are **done with the data cleaning**.

At this stage, the notebook has done its job: it helped us inspect the data, justify a few rules, and test the transformations interactively.

The next step is to convert the **stable cleaning logic** into reusable Python functions. That extraction step is the bridge from notebook work to production-friendly code.


Now refactor the cleaned workflow into functions and save them in `src/king_county_refactoring`.

This notebook focuses on **what should be refactored** and **why**. The companion notebook `king-county-data-preparation.ipynb` walks through the individual function extractions in more detail.

**Exercise goal:** Move the stable cleaning logic into reusable Python functions that can be imported elsewhere.

@TODO:
1. Implement functions for outlier cleanup, basement conversion, renovation-year transformation, and missing-value filling.
2. Write your implementation in `src/king_county_refactoring/data_preparation.py`.
3. Keep each function DataFrame-in/DataFrame-out and copy-first.
4. Chain the functions and verify that `view` and `waterfront` have no missing values.

**Hints:**
- Keep exploratory display code in the notebook, not in the final functions.
- Each function should own one transformation responsibility.
- Validate intermediate results after each step while developing.

> **Checkpoint:**
> You can chain all four functions without manual intermediate fixes.

Starter file: `src/king_county_refactoring/data_preparation.py`  
Reference solution: `src/king_county_refactoring/data_preparation_solution.py`


### **Quiz**

1. **Why might we want to refactor a Jupyter Notebook into a Python script?**  
    [ ] To make the notebook look shorter  
    [ ] For reusability and reproducibility in production environments  
    [ ] Because Python scripts always run faster than notebooks  
    [ ] To avoid using pandas and scikit-learn  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** For reusability and reproducibility in production environments  
      
      **Description:** Python scripts can be version-controlled, reused, and integrated into production workflows more easily than Jupyter Notebooks.
    </details>

---

2. **Which of the following is a limitation of keeping code only inside a Jupyter Notebook?**  
    [ ] Code cannot run on a laptop  
    [ ] It is harder to reuse and test code in larger projects  
    [ ] Notebooks cannot contain visualizations  
    [ ] Variables cannot be defined  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** It is harder to reuse and test code in larger projects  
      
      **Description:** While notebooks are great for exploration, scripts are better for modularity, testing, and integration with larger systems.
    </details>

---

3. **What is a common first step when refactoring a notebook into a script?**  
    [ ] Deleting all data analysis code  
    [ ] Moving reusable code (functions, preprocessing, etc.) into `.py` files  
    [ ] Changing all variables to uppercase  
    [ ] Removing markdown cells  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** Moving reusable code (functions, preprocessing, etc.) into `.py` files  
      
      **Description:** By extracting reusable parts (e.g., feature engineering, data loading) into scripts, we make them easier to import and maintain.
    </details>

---

4. **What advantage does separating data preparation code into a script provide?**  
    [ ] It prevents missing values  
    [ ] It allows the same transformations to be applied consistently across different projects or runs  
    [ ] It makes the code harder to read  
    [ ] It avoids the need for testing  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** It allows the same transformations to be applied consistently across different projects or runs  
      
      **Description:** A script ensures consistent and reproducible preprocessing, reducing errors from manual copy-pasting.
    </details>

---

5. **Who might be responsible for refactoring notebooks into production-ready scripts?**  
    [ ] Only software engineers  
    [ ] Only data scientists  
    [ ] Data scientists, software engineers, or machine learning engineers depending on the workflow  
    [ ] Only database administrators  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** Data scientists, software engineers, or machine learning engineers depending on the workflow  
      
      **Description:** Both data scientists and ML engineers often collaborate — data scientists create prototypes, and ML engineers refactor for production use.
    </details>

---

6. **Which of the following best describes the relationship between data science and machine learning engineering in this context?**  
    [ ] They are completely separate and never overlap  
    [ ] They overlap, especially in areas like code refactoring and reproducibility  
    [ ] Data science replaces machine learning engineering  
    [ ] Machine learning engineering is only about writing models  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** They overlap, especially in areas like code refactoring and reproducibility  
      
      **Description:** Data science focuses on insights and prototyping, while ML engineering ensures production readiness. Their work intersects in areas like refactoring.
    </details>
